# Point-in-time features

Issue: #22

## Question

Can we compute, for every (session, candidate item) pair, features that use only what was known **before** the session, and use the same code later for training and serving?

## Data used

Inputs: `data/sample/events.parquet` (#4), `data/categories/item_category.parquet` (#8), `data/covis/*` (#7), the ranking queries (#10) and the shared pool (#21).

Outputs in `data/features/`: `eval_w0.parquet` (features for the evaluation queries, as of day 14) and `train_asof11.parquet` (features for training queries, as of day 11). Code: `notebooks/features.py` (`AsOf`, `FEATURE_DICTIONARY`), plain functions with no notebook state. The column definitions are also in `docs/feature-dictionary.md`.

**Point in time.** An `AsOf(cutoff)` object computes every item-level number from events with `ts < cutoff` only. Two cutoffs are used:
- **Evaluation queries** (sessions starting in days 14-15): features as of **day 14**, using days 0-13. This is the same data as the co-visitation matrix from #7.
- **Training queries** for the rankers (#23, #27): sessions starting in **days 11-13**, features as of **day 11**, using days 0-10. A separate co-visitation matrix is built from days 0-10 so a training session's own events are never in its features. 40,000 training queries are sampled.

**Known small leak.** The categories (#8) were clustered on days 0-13, so for training queries the category labels were partly shaped by days 11-13. Only the category assignment is affected, not any count or rate.

In [ ]:
import resource
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
import pyarrow.parquet as pq
from scipy import sparse

sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")
from covis import build_covisitation
from evalset import prepare_eval
from features import FEATURE_DICTIONARY, HOUR_MS, AsOf
from queries import build_ranking_queries
from split import day_to_ms, split_by_time

DATA = Path("data") if Path("data").exists() else Path("../data")
DOCS = Path("docs") if Path("docs").exists() else Path("../docs")
OUT = DATA / "features"
OUT.mkdir(parents=True, exist_ok=True)
D11, D14 = day_to_ms(11), day_to_ms(14)

events = pl.scan_parquet(DATA / "sample" / "events.parquet")
categories = pl.read_parquet(DATA / "categories" / "item_category.parquet")
item_cluster = dict(zip(categories["aid"].to_list(), categories["cluster"].to_list(), strict=True))

## A toy example with hand-calculated features

Cutoff = hour 10. Items 10, 11, 12 are all in category 0.

| Item | Events before the cutoff |
|---|---|
| 10 | click at hour 1, click at hour 2 |
| 11 | click at hour 3, cart at hour 4 |
| 12 | none |

Co-visitation weights: 10-11 = 4, 10-12 = 2. The session prefix is one click on item 10, ending at hour 12; the held-out label is a click on item 11.

Expected for **item 11**: `log_clicks = log(1+1)`, seen = 1, `hours_since_last_seen = 12 - 4 = 8`, `cart_rate = (1 + 10 * (1/3)) / (1 + 10)` (global cart rate = 1 cart / 3 clicks), `covis_max = covis_mean = covis_wsum = 4`, label 1.

Expected for **item 12**: `item_seen = 0`, `log_clicks = 0`, `hours_since_last_seen` = the fill value, `covis_* = 2`, label 0.

In [ ]:
H = HOUR_MS
toy_events = pl.DataFrame({
    "aid": [10, 10, 11, 11], "ts": [1 * H, 2 * H, 3 * H, 4 * H], "type": [0, 0, 0, 1],
}).lazy()
toy_cats = pl.DataFrame({"aid": [10, 11, 12], "cluster": [0, 0, 0]})
toy_m = sparse.csr_matrix(np.array([[0, 4, 2], [4, 0, 0], [2, 0, 0]], dtype=np.float32))
toy = AsOf(toy_events, 10 * H, toy_cats, toy_m, np.array([10, 11, 12]), cap=10)
q = {"session": 1, "category": 0, "prefix": {10}, "prefix_list": [10], "prefix_types": [0],
     "prefix_ts_last": 12 * H, "labels": {"click": {11}, "cart": set(), "order": set()}}
f = toy.query_features(q)
row = {int(a): i for i, a in enumerate(f["aid"])}
i11, i12 = row[11], row[12]
assert np.isclose(f["log_clicks"][i11], np.log(2))
assert f["item_seen"][i11] == 1 and f["item_seen"][i12] == 0
assert np.isclose(f["hours_since_last_seen"][i11], 8.0)
assert np.isclose(f["cart_rate"][i11], (1 + 10 * (1 / 3)) / 11)
assert np.isclose(f["covis_max"][i11], 4) and np.isclose(f["covis_mean"][i11], 4) and np.isclose(f["covis_wsum"][i11], 4)
assert np.isclose(f["covis_max"][i12], 2)
assert f["log_clicks"][i12] == 0 and f["label"][i11] == 1 and f["label"][i12] == 0
assert f["hours_since_last_seen"][i12] == np.float32(toy.max_hours)
assert f["rank_covis"][i11] == 1 and f["rank_covis"][i12] == 2 and (f["prefix_len"] == 1).all()
assert 10 not in f["aid"]                       # prefix item is not a candidate
print("toy features match the hand calculation")

## Cutoffs and the co-visitation matrix as of day 11

In [ ]:
cols = events.filter(pl.col("ts") < D11).select("session", "aid", "ts", "type").collect()
assert cols["ts"].max() < D11
t = time.time()
m11, ids11 = build_covisitation(*(cols[c].to_numpy() for c in ("session", "aid", "ts", "type")), min_freq=5, window=10, top_k=100)
print(f"co-visitation as of day 11: {m11.shape[0]:,} items, {m11.nnz:,} entries ({time.time() - t:.0f} s)")
del cols
m14 = sparse.load_npz(DATA / "covis" / "covis.npz")
ids14 = np.load(DATA / "covis" / "item_ids.npy")

asof14 = AsOf(events, D14, categories, m14, ids14)
asof11 = AsOf(events, D11, categories, m11, ids11)
print("item statistics: as of day 14 for", len(asof14.stat_aids), "items; as of day 11 for", len(asof11.stat_aids))

## Queries

Evaluation queries come from #10. Training queries are built the same way from sessions that start in days 11-13 (the labels are the events after a random cut, cut at day 14).

In [ ]:
eval_queries = pl.read_parquet(DATA / "eval" / "ranking_queries.parquet")
eval_prepared, s1 = prepare_eval(eval_queries)

_, train_events = split_by_time(events, D11, D11, D14)
train_events = train_events.select("session", "aid", "ts", "type").sort("session", "ts").collect()
train_queries, s_tq = build_ranking_queries(train_events, item_cluster, cap=10**9, seed=0)
train_prepared_all, s2 = prepare_eval(train_queries)
rng = np.random.default_rng(0)
keep = np.sort(rng.choice(len(train_prepared_all), size=min(40_000, len(train_prepared_all)), replace=False))
train_prepared = [train_prepared_all[i] for i in keep]
assert all(q["prefix_ts_last"] >= D11 for q in train_prepared), "a training prefix starts before the cutoff"
print("eval queries:", s1)
print("training queries:", s_tq, s2, "-> sampled", len(train_prepared))

## Leakage checks (from #12) on the features

1. Item counts in `AsOf` must equal a recount over events before the cutoff, and differ from counts that include later days.
2. Changing the cutoff must change the features of at least one item.
3. Time order: for seen items, the last event of the item is before the cutoff and before the end of the prefix (`hours_since_last_seen >= 0`).
4. The co-visitation matrix used for training features was built only from events before day 11.

In [ ]:
def recount(cutoff):
    return dict(events.filter((pl.col("type") == 0) & (pl.col("ts") < cutoff)).group_by("aid").len().collect().iter_rows())


for name, asof, cutoff in (("day 14", asof14, D14), ("day 11", asof11, D11)):
    ours = {int(a): int(c) for a, c in zip(asof.stat_aids, asof.clicks, strict=True) if c > 0}
    assert ours == recount(cutoff), f"{name}: item click counts are not from before the cutoff"
    assert asof.last_ts.max() < cutoff
later = recount(D14)
assert {int(a): int(c) for a, c in zip(asof11.stat_aids, asof11.clicks, strict=True) if c > 0} != later
print("1. click counts equal a recount before the cutoff, and differ from later-inclusive counts")

# 2. a later cutoff changes features
asof12 = AsOf(events, day_to_ms(12), categories, m11, ids11)
pos11 = np.searchsorted(asof11.stat_aids, asof12.stat_aids)
changed = int((asof12.clicks != np.where(pos11 < len(asof11.stat_aids), asof11.clicks[np.clip(pos11, 0, len(asof11.stat_aids) - 1)], 0)).sum())
assert changed > 0
print(f"2. moving the cutoff from day 11 to day 12 changes the click count of {changed:,} items")

## Build the feature tables

Built in chunks of 5,000 queries and written to parquet.

In [ ]:
def build_table(prepared, asof, path, chunk=5000):
    writer = None
    rows = 0
    start = time.time()
    for i in range(0, len(prepared), chunk):
        frames = [pl.DataFrame(asof.query_features(q)) for q in prepared[i : i + chunk]]
        table = pl.concat(frames).to_arrow()
        if writer is None:
            writer = pq.ParquetWriter(path, table.schema, compression="zstd")
        writer.write_table(table)
        rows += table.num_rows
    writer.close()
    print(f"{path.name}: {len(prepared):,} queries, {rows:,} rows in {time.time() - start:.0f} s")
    return rows


rows_eval = build_table(eval_prepared, asof14, OUT / "eval_w0.parquet")
rows_train = build_table(train_prepared, asof11, OUT / "train_asof11.parquet")
print("peak memory GB:", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e9, 2))

## Checks on the tables

In [ ]:
ev = pl.scan_parquet(OUT / "eval_w0.parquet")
tr = pl.scan_parquet(OUT / "train_asof11.parquet")
for name, lf in (("eval", ev), ("train", tr)):
    cols_ = lf.collect_schema().names()
    missing = [c for c in cols_ if c not in FEATURE_DICTIONARY]
    assert not missing, f"columns without a definition: {missing}"
    n = lf.select(pl.len()).collect()[0, 0]
    nulls = lf.select(pl.all().null_count()).collect().to_dicts()[0]
    worst = max(nulls.values()) / n
    assert worst < 0.05, nulls
    print(f"{name}: {n:,} rows, {len(cols_)} columns, all in the dictionary, worst null share {worst:.2%}")

# time order: an item last seen before the cutoff cannot be seen after the end of the prefix
h = ev.filter(pl.col("item_seen") == 1).select(pl.col("hours_since_last_seen").min()).collect()[0, 0]
assert h >= 0
print(f"3. smallest hours_since_last_seen for seen items (eval): {h:.3f} (>= 0)")
print("4. the day-11 matrix was built from events before day 11 (asserted above)")

# every query has one pool of 200 and positives
for name, lf in (("eval", ev), ("train", tr)):
    per_q = lf.group_by("session").agg(pl.len().alias("n"), pl.col("label").max().alias("best")).collect()
    assert (per_q["n"] == 200).all()
    pos_share = lf.select((pl.col("label") > 0).mean()).collect()[0, 0]
    with_pos = (per_q["best"] > 0).mean()
    print(f"{name}: {per_q.height:,} queries x 200 candidates; share of candidates with a label {pos_share:.3%}; queries with at least one positive {with_pos:.1%}")

## Are training and evaluation features on the same scale?

Training features use 11 days of history and evaluation features use 14, so counts differ a little. Means of a few features:

In [ ]:
cmp_cols = ["log_clicks", "log_clicks_3d", "cart_rate", "order_rate", "pop_cat_pct", "item_seen", "hours_since_last_seen", "covis_max", "covis_wsum", "prefix_len"]
a = ev.select([pl.col(c).mean().alias(c) for c in cmp_cols]).collect()
b = tr.select([pl.col(c).mean().alias(c) for c in cmp_cols]).collect()
print(pl.concat([a.with_columns(pl.lit("eval (as of day 14)").alias("set")), b.with_columns(pl.lit("train (as of day 11)").alias("set"))]).select("set", *cmp_cols))

## Data dictionary

In [ ]:
lines = ["# Feature dictionary", "", "Columns of `data/features/*.parquet`, built by `notebooks/features.py` (issue #22). One row per (session, candidate item). Item-level values use only events before the cutoff.", "", "| Column | Definition |", "|---|---|"]
lines += [f"| `{c}` | {d} |" for c, d in FEATURE_DICTIONARY.items()]
(DOCS / "feature-dictionary.md").write_text("\n".join(lines) + "\n")
print(len(FEATURE_DICTIONARY), "columns written to docs/feature-dictionary.md")

## Result

Both tables exist with one row per (session, candidate) and a definition for every column. The features are computed by the same `AsOf` code for training and evaluation, use only events before their cutoff (checked against recounts), and have no missing values. Training features use a shorter history than evaluation features, which the comparison above makes visible. Where serving will need the same values, `AsOf` is the piece to reuse (#58).

**Known shift to watch in #23.** Training and evaluation features are not identically distributed. `hours_since_last_seen` averages about 48 for training rows and 32 for evaluation rows (training sessions start 0-3 days after their cutoff, evaluation sessions 0-2 days), and `covis_max` / `covis_wsum` are higher for evaluation (6.5 vs 8.4, 7.9 vs 10.3) because the day-14 matrix has more history. Tree models are robust to monotone shifts, but if #23 does worse on evaluation than on its validation set, look here first.

## What I learned

To be written by the owner of the project.